In [ ]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
RAW_PATH = ROOT / "data" / "raw" / "adult24.csv"
OUT_PATH = ROOT / "data" / "processed" / "lykke_clean.csv"

df = pd.read_csv(RAW_PATH, na_values=["", " ", "-", "..", ":", "ukjent", "unknown"])
df = df[(df["SCHCURENR_A"] == 1) & (df["AGEP_A"].between(18, 29))]
df_lykke = df[["RATCAT_A", "FDSCAT4_A", "HOUSECOST_A", "HYPEV_A", "ASEV_A", "ARTHEV_A", "DIBEV_A"]].copy()

In [ ]:
df_lykke.head()

In [ ]:
df_lykke.nunique()

In [ ]:
df_lykke["RATCAT_A"].value_counts()

In [ ]:
#"Ingen dårlige verdier ifølge Frequency Codebook"

In [ ]:
df_lykke["FDSCAT4_A"].value_counts()

In [ ]:
#"4 dårlige verdier ifølge frequency codebook"

In [ ]:
df_lykke["HOUSECOST_A"].value_counts()

In [ ]:
#"11 dårlige verdier"

In [ ]:
df_lykke["HYPEV_A"].value_counts()

In [ ]:
#"Ingen feil verdier"

In [ ]:
df_lykke["ASEV_A"].value_counts()

In [ ]:
#"1 dårlig verdi"

In [ ]:
df_lykke["ARTHEV_A"].value_counts()

In [ ]:
#1 dårlig verdi

In [ ]:
df_lykke["DIBEV_A"].value_counts()

In [ ]:
#"Ingen dårlige verdier"

In [ ]:
import numpy as np

bad_values = ["FDSCAT4_A", "HOUSECOST_A", "HYPEV_A", "ASEV_A", "ARTHEV_A", "DIBEV_A"]
yes_no = ["HOUSECOST_A", "HYPEV_A", "ASEV_A", "ARTHEV_A", "DIBEV_A"]

for column in bad_values:
    df_lykke[column] = df_lykke[column].replace({7: np.nan, 8: np.nan, 9: np.nan})
for column in yes_no:
    df_lykke[column] = df_lykke[column].replace({1: "YES", 2: "NO"})

df_lykke = df_lykke.rename(columns={
    "RATCAT_A": "family_income",
    "FDSCAT4_A": "food_security",
    "HOUSECOST_A": "housing_cost",
    "HYPEV_A": "high_blpressure",
    "ASEV_A": "had_asthma",
    "ARTHEV_A": "had_arthritis",
    "DIBEV_A": "had_diabetes"
})

df_lykke = df_lykke.reset_index(drop=True)

print(df_lykke)

In [ ]:
print(f"Prepared {len(df_lykke)} rows")

In [ ]:
df_lykke.head()

In [ ]:
df_lykke.info()

In [ ]:
df_lykke.isna().sum()

In [ ]:
df_lykke["housing_cost"].value_counts(dropna=False)

In [ ]:
def food_security(value):
    if value == 1:
        return "High"
    elif value == 2:
        return "Marginal"
    elif value == 3:
        return "Low"
    elif value == 4:
        return "Very Low"
    else: 
        return np.nan

df_lykke_fsl = df_lykke.copy()    

df_lykke_fsl["food_security_level"] = df_lykke_fsl["food_security"].map(food_security)
df_lykke = df_lykke_fsl
df_lykke.to_csv(OUT_PATH, index=False)
print(f"Saved {len(df_lykke)} rows to {OUT_PATH}")
df_lykke.head()

In [ ]:
#Dette kan også gjøres hvor jeg legger den nye kolonnen ved siden av food_security kolonnen

In [ ]:
df_lykke_test = df_lykke.copy()

df_lykke_test["food_security"] = df_lykke_test["food_security"].replace({1.0: "High", 2.0: "Marginal", 3.0: "Low", 4.0: "Very low"})

df_lykke_test.head()

In [ ]:
# Save the final cleaned dataset after all transformations.
if "food_security_level" not in df_lykke.columns:
    df_lykke["food_security_level"] = df_lykke["food_security"].map(food_security)

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_lykke.to_csv(OUT_PATH, index=False)
print(f"Saved {len(df_lykke)} rows and {len(df_lykke.columns)} columns to {OUT_PATH}")

In [ ]:
#Dette er et annet forslag, hvor de originale verdiene til food_security byttes ut med mer lesbar tekst, i dette tilfellet; high, marginal, low, very low